# Crawling Berita Detik.com
* Berita Sport : 100 
* Berita Finance : 100

In [2]:
%pip install trafilatura requests beautifulsoup4 pandas ipykernel

  Using cached trafilatura-2.2.0-py3-none-any.whl.metadata (13 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached pandas-3.0.5-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached charset_normalizer-3.5.1-cp314-cp314-win_amd64.whl.metadata (46 kB)
  Using cached courlan-1.4.0-py3-none-any.whl.metadata (18 kB)
  Using cached htmldate-1.10.0-py3-none-any.whl.metadata (9.8 kB)
  Using cached justext-3.0.2-py2.py3-none-any.whl.metadata (7.3 kB)
  Using cached lxml-6.1.3-cp314-cp314-win_amd64.whl.metadata (3.4 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached idna-3.19-py3-none-any.whl.metadata (9.2 kB)
  Using cached soupsieve-2.9.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached babel-2.18.0-py3-none-any.whl.met

## Import Library

In [3]:
import requests
import trafilatura
import pandas as pd
import time
import re

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from IPython.display import display

## Konfigurasi

In [4]:
TARGET_PER_LABEL = 100

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/152.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;"
        "q=0.9,image/avif,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
    "Connection": "keep-alive"
}

session = requests.Session()
session.headers.update(HEADERS)

## Membersihkan Teks

In [5]:
def clean_text(text):
    if not text:
        return None

    # Hilangkan enter, tab, dan spasi berlebihan
    text = re.sub(r"\s+", " ", text)

    return text.strip()

## Cek URL 

In [6]:
def is_article_url(url):
    """
    Mengecek apakah URL merupakan URL artikel Detik.
    """

    if not url:
        return False

    url_lower = url.lower()

    # Harus dari domain detik.com
    if "detik.com" not in url_lower:
        return False

    # URL artikel Detik umumnya memiliki /d-angka
    if not re.search(r"/d-\d+", url_lower):
        return False

    # URL yang bukan artikel
    blacklist = [
        "/foto/",
        "/video/",
        "/detiktv/",
        "/tag/",
        "/search/",
        "/live/",
        "/infografis/"
    ]

    if any(item in url_lower for item in blacklist):
        return False

    return True

## Mengambil link artikel dan pagination

In [7]:
def get_links_from_index(index_url):
    """
    Mengambil:
    - URL artikel
    - URL pagination berikutnya
    """

    try:

        response = session.get(
            index_url,
            timeout=30
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        article_links = []
        pagination_links = []

        # =====================================================
        # AMBIL SEMUA LINK
        # =====================================================

        for a in soup.find_all("a", href=True):

            href = a.get("href")

            if not href:
                continue

            url = urljoin(index_url, href)

            # Hilangkan query dan fragment
            url = url.split("?")[0]
            url = url.split("#")[0]

            # =================================================
            # LINK ARTIKEL
            # =================================================

            if is_article_url(url):

                if url not in article_links:
                    article_links.append(url)

                continue

            # =================================================
            # LINK PAGINATION
            # =================================================

            url_lower = url.lower()

            if (
                "detik.com" in url_lower
                and "/indeks" in url_lower
            ):

                # Jangan masukkan halaman yang sama
                if url != index_url:

                    if url not in pagination_links:
                        pagination_links.append(url)

        return article_links, pagination_links

    except Exception as e:

        print(f"❌ Gagal mengambil index:")
        print(index_url)
        print("Error:", e)

        return [], []

## Ekstraksi artikel menggunakan Trafilatura

In [8]:
def extract_article(url):
    """
    Mengambil isi utama artikel menggunakan Trafilatura.
    """

    try:

        response = session.get(
            url,
            timeout=30
        )

        response.raise_for_status()

        html = response.text

        # Ekstraksi menggunakan Trafilatura
        text = trafilatura.extract(
            html,
            include_comments=False,
            include_tables=False,
            include_links=False,
            favor_precision=True
        )

        text = clean_text(text)

        # Artikel minimal 200 karakter
        if text and len(text) >= 200:
            return text

        return None

    except Exception as e:

        print("❌ Gagal mengambil artikel")
        print(url)
        print("Error:", e)

        return None

In [22]:
def crawl_category(
    start_url,
    label,
    target=100,
    max_pages=100
):
    """
    Crawling artikel berdasarkan kategori.

    Parameter:
    start_url  : URL indeks awal
    label      : sport / finance
    target     : jumlah artikel yang diinginkan
    max_pages  : maksimum halaman indeks yang dikunjungi
    """

    data = []

    visited_pages = set()
    collected_urls = set()

    # Halaman pertama
    current_url = start_url

    page_count = 0

    print("=" * 70)
    print(f"CRAWLING {label.upper()}")
    print(f"TARGET : {target} ARTIKEL")
    print("=" * 70)

    while (
        len(data) < target
        and page_count < max_pages
    ):

        page_count += 1

        # =====================================================
        # CEK HALAMAN SUDAH PERNAH DIKUNJUNGI
        # =====================================================

        if current_url in visited_pages:

            print("⚠ Halaman sudah dikunjungi.")
            break

        visited_pages.add(current_url)

        print()
        print("-" * 70)
        print(f"INDEX HALAMAN {page_count}")
        print(current_url)
        print("-" * 70)

        # =====================================================
        # AMBIL LINK
        # =====================================================

        article_links, pagination_links = get_links_from_index(
            current_url
        )

        print(
            f"Ditemukan {len(article_links)} "
            "calon artikel."
        )

        # =====================================================
        # JIKA TIDAK ADA ARTIKEL
        # =====================================================

        if len(article_links) == 0:

            print("⚠ Tidak ada artikel pada halaman ini.")

        # =====================================================
        # PROSES ARTIKEL
        # =====================================================

        for url in article_links:

            if len(data) >= target:
                break

            # Hindari duplikat URL
            if url in collected_urls:
                continue

            collected_urls.add(url)

            nomor = len(data) + 1

            print(
                f"[{nomor}/{target}] "
                "Ekstraksi...",
                end=" "
            )

            text = extract_article(url)

            if text:

                data.append({
                    "isi_berita": text,
                    "label": label,
                    "link_berita": url
                })

                print("✓ berhasil")

            else:

                print("✗ gagal")

            # Delay supaya tidak terlalu agresif
            time.sleep(1.5)

        # =====================================================
        # CEK TARGET
        # =====================================================

        if len(data) >= target:

            print()
            print("🎯 TARGET TERCAPAI!")
            break

        # =====================================================
        # CARI HALAMAN BERIKUTNYA
        # =====================================================

        if not pagination_links:

            print()
            print(
                "⚠ Tidak ditemukan pagination berikutnya."
            )

            break

        # =====================================================
        # PILIH PAGINATION
        # =====================================================

        next_url = None

        # Prioritaskan URL yang belum dikunjungi
        for link in pagination_links:

            if link not in visited_pages:
                next_url = link
                break

        if not next_url:

            print(
                "⚠ Semua pagination sudah dikunjungi."
            )

            break

        current_url = next_url

    # =========================================================
    # HASIL AKHIR
    # =========================================================

    print()
    print("=" * 70)
    print(f"CRAWLING {label.upper()} SELESAI")
    print(f"Berhasil mendapatkan : {len(data)} artikel")
    print(f"Halaman dikunjungi   : {page_count}")
    print("=" * 70)

    return data

In [25]:
sport_data = crawl_category(
    start_url="https://sport.detik.com/sport-lain/indeks",
    label="sport",
    target=100,
    max_pages=100
)

CRAWLING SPORT
TARGET : 100 ARTIKEL

----------------------------------------------------------------------
INDEX HALAMAN 1
https://sport.detik.com/sport-lain/indeks
----------------------------------------------------------------------
Ditemukan 20 calon artikel.
[1/100] Ekstraksi... ✓ berhasil
[2/100] Ekstraksi... ✓ berhasil
[3/100] Ekstraksi... ✓ berhasil
[4/100] Ekstraksi... ✓ berhasil
[5/100] Ekstraksi... ✓ berhasil
[6/100] Ekstraksi... ✓ berhasil
[7/100] Ekstraksi... ✓ berhasil
[8/100] Ekstraksi... ✓ berhasil
[9/100] Ekstraksi... ✓ berhasil
[10/100] Ekstraksi... ✓ berhasil
[11/100] Ekstraksi... ✓ berhasil
[12/100] Ekstraksi... ✓ berhasil
[13/100] Ekstraksi... ✓ berhasil
[14/100] Ekstraksi... ✓ berhasil
[15/100] Ekstraksi... ✓ berhasil
[16/100] Ekstraksi... ✓ berhasil
[17/100] Ekstraksi... ✓ berhasil
[18/100] Ekstraksi... ✓ berhasil
[19/100] Ekstraksi... ✓ berhasil
[20/100] Ekstraksi... ✓ berhasil

----------------------------------------------------------------------
INDEX HALAMA

In [11]:
print("Jumlah berita sport:", len(sport_data))

Jumlah berita sport: 100


Crawling Finance

In [28]:
finance_data = crawl_category(
    start_url="https://finance.detik.com/berita-ekonomi-bisnis/indeks",
    label="finance",
    target=100,
    max_pages=100
)

CRAWLING FINANCE
TARGET : 100 ARTIKEL

----------------------------------------------------------------------
INDEX HALAMAN 1
https://finance.detik.com/berita-ekonomi-bisnis/indeks
----------------------------------------------------------------------
Ditemukan 20 calon artikel.
[1/100] Ekstraksi... ✓ berhasil
[2/100] Ekstraksi... ✓ berhasil
[3/100] Ekstraksi... ✓ berhasil
[4/100] Ekstraksi... ✓ berhasil
[5/100] Ekstraksi... ✓ berhasil
[6/100] Ekstraksi... ✓ berhasil
[7/100] Ekstraksi... ✓ berhasil
[8/100] Ekstraksi... ✓ berhasil
[9/100] Ekstraksi... ✓ berhasil
[10/100] Ekstraksi... ✓ berhasil
[11/100] Ekstraksi... ✓ berhasil
[12/100] Ekstraksi... ✓ berhasil
[13/100] Ekstraksi... ✓ berhasil
[14/100] Ekstraksi... ✓ berhasil
[15/100] Ekstraksi... ✓ berhasil
[16/100] Ekstraksi... ✓ berhasil
[17/100] Ekstraksi... ✓ berhasil
[18/100] Ekstraksi... ✓ berhasil
[19/100] Ekstraksi... ✓ berhasil
[20/100] Ekstraksi... ✓ berhasil

--------------------------------------------------------------------

In [13]:
print("Jumlah berita finance:", len(finance_data))

Jumlah berita finance: 100


In [29]:
all_data = sport_data + finance_data

df = pd.DataFrame(all_data)

print("Jumlah data:", len(df))

Jumlah data: 200


In [30]:
df_sport = (
    df[df["label"] == "sport"]
    .drop_duplicates(subset=["isi_berita"])
    .head(100)
)

df_finance = (
    df[df["label"] == "finance"]
    .drop_duplicates(subset=["isi_berita"])
    .head(100)
)

df = pd.concat(
    [
        df_sport,
        df_finance
    ],
    ignore_index=True
)

In [31]:
id_values = (
    df["label"].map({"sport": 1, "finance": 101})
    + df.groupby("label").cumcount()
)

if "id" in df.columns:
    df["id"] = id_values
else:
    df.insert(0, "id", id_values)

In [17]:
print("=" * 60)
print("HASIL DATASET")
print("=" * 60)

print("Total data :", len(df))

print("\nJumlah berdasarkan label:")
print(df["label"].value_counts())

print("\nRentang ID:")
print(df["id"].min(), "-", df["id"].max())

HASIL DATASET
Total data : 200

Jumlah berdasarkan label:
label
sport      100
finance    100
Name: count, dtype: int64

Rentang ID:
1 - 200


In [18]:
print("Jumlah isi berita kosong:")
print(df["isi_berita"].isna().sum())

print("\nJumlah isi berita < 200 karakter:")
print(
    (df["isi_berita"].str.len() < 200).sum()
)

Jumlah isi berita kosong:
0

Jumlah isi berita < 200 karakter:
0


In [33]:
filename = "dataset_detik_sport_finance_final.xlsx"

df.to_excel(
    filename,
    index=False
)

print(f"✓ Dataset berhasil disimpan sebagai: {filename}")

✓ Dataset berhasil disimpan sebagai: dataset_detik_sport_finance_final.xlsx
